# Copilot Agent Transcript — Parser (Fabric, dual-source)

Reads Copilot Studio `ConversationTranscript` rows, parses the activity JSON, and
writes a normalised set of Delta tables to the Lakehouse for the Power BI dashboard.

**Three source modes — pick one in cell 1 (`SOURCE_MODE`):**

| Mode | When to use | Reads from |
|---|---|---|
| `dataverse` | Live pull from one or more Copilot Studio environments | Dataverse Web API (needs app registration + Application User) |
| `lakehouse` | Re-parse the already-landed raw table (e.g. after a parser update) | `dbo.conversationtranscripts_raw` Delta table |
| `files` | One-off / CI smoke test on a sample CSV | `Files/copilot_transcripts/conversationtranscripts.csv` |

**Outputs (7 Delta tables, schema `dbo`):**

| Table | Grain |
|---|---|
| `conversationtranscripts_raw` | raw landing — one row per transcript (dataverse mode only; set `RAW_TABLE=''` to skip) |
| `agent_sessions` | one row per conversation (includes 4 enrichment columns) |
| `agent_turns` | one row per user/bot message (with topic + role enrichment) |
| `agent_errors` | one row per `ErrorTraceData` event |
| `agent_subagents` | one row per `ConnectedAgent` invocation |
| `agent_catalogue` | agent dimension (one row per `botSchemaName`) |
| `agent_performance` | per-conversation KPI fact (includes 18 enrichment columns) |
| `agent_variables` | one row per (conversation, variable_name) — decomposes `VariableAssignment` activities |

**Safety net:** if the ingest yields **zero rows**, the write step raises before
touching the existing tables — so a transient Dataverse failure can never wipe a
good Lakehouse.

**Permissions (`SOURCE_MODE='dataverse'`)** — an Entra app registration added as an
**Application User** in each target environment, with a security role granting
**Organization-scope** read on `conversationtranscript` (User-scope returns 0 rows
even when transcripts exist). `lakehouse` and `files` modes need no Dataverse auth.


## 1. Configuration  ·  *(mark this as the pipeline `parameters` cell)*

In [ ]:
# === CONFIG ===  (tag this cell as the pipeline `parameters` cell)

# --- Source mode ---
# 'dataverse'  -> live Web API pull from each environment in DATAVERSE_ENVIRONMENTS
# 'lakehouse'  -> re-parse the already-landed dbo.conversationtranscripts_raw table
# 'files'      -> read a CSV at TRANSCRIPTS_FILE (used for CI / smoke tests)
SOURCE_MODE   = 'dataverse'

# --- Dataverse environment(s) — used when SOURCE_MODE='dataverse' ---
# ONE environment:   set DATAVERSE_URL and leave DATAVERSE_URLS = [].
# MANY environments: set DATAVERSE_URLS to a list of env URLs (DATAVERSE_URL is then ignored).
#   The app registration must be added as an Application User in EVERY listed environment.
DATAVERSE_URL  = 'https://orgXXXXXXXX.crm.dynamics.com'   # single environment URL, NO trailing slash
DATAVERSE_URLS = [
    # 'https://org1.crm.dynamics.com',
    # 'https://org2.crm.dynamics.com',
]
TAG_ENVIRONMENT = True   # add a 'SourceEnvironment' column to every row (lets the PBIP slice by env)

TENANT_ID     = '<your-tenant-guid>'
CLIENT_ID     = '<your-app-reg-client-id>'
CLIENT_SECRET = '<your-client-secret-value>'   # prod: notebookutils.credentials.getSecret('<kv-uri>', '<name>')
LOOKBACK_DAYS = 7        # incremental window on conversationstarttime; 0 = full pull (all rows)
PAGE_SIZE     = 5000     # Dataverse OData max page size (server caps at 5000)
RAW_TABLE     = 'dbo.conversationtranscripts_raw'   # raw landing Delta table written by dataverse mode ('' to skip landing)

# --- Lakehouse re-parse source — used when SOURCE_MODE='lakehouse' ---
RAW_TABLE_INPUT = 'dbo.conversationtranscripts_raw'   # read this table back, re-parse, rewrite the fact tables

# Resolve the environment list: DATAVERSE_URLS wins if non-empty, else fall back to the single URL.
# Placeholder/empty entries are dropped so an unset DATAVERSE_URL doesn't get pulled by accident.
DATAVERSE_ENVIRONMENTS = [
    u.rstrip('/') for u in (DATAVERSE_URLS if DATAVERSE_URLS else [DATAVERSE_URL])
    if u and not u.startswith('<') and 'XXXX' not in u
]

# --- Files (CI / smoke test transcripts) ---
SOURCE_DIR        = 'Files/copilot_transcripts'
TRANSCRIPTS_FILE  = f'{SOURCE_DIR}/conversationtranscripts.csv'   # used only when SOURCE_MODE='files'

# --- Output ---
OUTPUT_PREFIX = 'dbo'        # Delta schema. Tables: dbo.agent_sessions, dbo.agent_turns, ...
WRITE_MODE    = 'overwrite'  # 'overwrite' = full snapshot; 'append' = add rows; 'merge' = upsert on natural keys
TEXT_TRUNCATE = 500          # max chars kept for free-text fields (PII-safe preview)

# --- Zero-row safeguard ---
# If True, the write step raises BEFORE touching existing tables when the ingest
# produced zero parsed rows. Prevents a transient Dataverse permission failure
# (e.g. User-scope role on `conversationtranscript`) from blanking the Lakehouse.
# Set to False only for an intentional reset.
ABORT_ON_EMPTY = True

# --- Transcript enrichment: topics + role inference (5b, non-destructive) ---
TOPIC_METHOD   = 'keyword'   # 'keyword' = built-in taxonomy (no deps); 'llm' = Azure OpenAI tagging (set AOAI_* below)
ROLE_INFERENCE = True        # add 'role_inferred' + 'role_mismatch' from message-content heuristics

# Azure OpenAI — only used when TOPIC_METHOD = 'llm'
AOAI_ENDPOINT   = ''
AOAI_DEPLOYMENT = 'gpt-4o-mini'
AOAI_API_KEY    = ''
AOAI_API_VERSION = '2024-06-01'

# Business-topic taxonomy used by both the keyword and LLM classifiers. Edit to fit the customer.
TOPIC_TAXONOMY = [
    'Leave & Family', 'Pay & Bonus', 'Benefits & Perks', 'Recruitment & Hiring',
    'Onboarding & Joiner', 'Access & IT Support', 'Software & Licensing',
    'Contracts & Policy', 'Facilities & Workplace', 'People & Directory',
    'Field & Technical', 'Other',
]


## 1b. Preflight  ·  *(run this FIRST in `dataverse` mode — fails fast with a clear verdict)*

In [ ]:
# === 1b. PREFLIGHT (Dataverse auth + permission check) ===
# Skipped silently when SOURCE_MODE is 'files' or 'lakehouse' — no Dataverse auth needed there.
if SOURCE_MODE == 'files':
    print("SOURCE_MODE='files' — preflight skipped (CSV will be read in cell 2b).")
elif SOURCE_MODE == 'lakehouse':
    print(f"SOURCE_MODE='lakehouse' — preflight skipped (re-parsing {RAW_TABLE_INPUT}).")
elif not DATAVERSE_ENVIRONMENTS:
    raise SystemExit("✗ No valid environments. Set DATAVERSE_URL or DATAVERSE_URLS in CONFIG "
                     "(placeholder/<...>/XXXX values are ignored).")
else:
    import requests

    def _preflight_one(base):
        """Returns (ok: bool, message: str) for a single environment URL."""
        base = base.rstrip('/')
        try:
            t = requests.post(
                f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token',
                data={'grant_type': 'client_credentials', 'client_id': CLIENT_ID,
                      'client_secret': CLIENT_SECRET, 'scope': f'{base}/.default'},
                timeout=60)
        except Exception as e:
            return False, f'token request failed to send: {e}'
        if t.status_code != 200:
            err = t.json().get('error', '?') if t.headers.get('content-type','').startswith('application/json') else t.text[:200]
            return False, f"token HTTP {t.status_code}: {err}  (check TENANT_ID/CLIENT_ID/CLIENT_SECRET; 'invalid_client' = bad secret)"
        tok = t.json()['access_token']
        try:
            r = requests.get(
                f'{base}/api/data/v9.2/conversationtranscripts?$select=conversationtranscriptid&$top=1',
                headers={'Authorization': f'Bearer {tok}', 'Accept': 'application/json',
                         'OData-MaxVersion': '4.0', 'OData-Version': '4.0'}, timeout=60)
        except Exception as e:
            return False, f'data GET failed to send: {e}'
        if r.status_code == 403:
            return False, ("403 Forbidden — token valid but app has no access HERE. Add it as an "
                           "Application User in THIS environment (Power Platform Admin Center → "
                           "Settings → Users + permissions → Application users) with READ on conversationtranscripts.")
        if r.status_code == 404:
            return False, "404 — wrong URL (must be https://<org>.crm.dynamics.com, no trailing slash)."
        if r.status_code != 200:
            return False, f'HTTP {r.status_code}: {r.text[:200]}'
        n = len(r.json().get('value', []))
        if n == 0:
            return True, ('read OK but 0 rows on $top=1 — env may have no transcripts, OR the app user has '
                          'a USER-scope read role rather than ORGANIZATION-scope. Check the security role on the app user.')
        return True, 'read OK'

    print(f'Preflight across {len(DATAVERSE_ENVIRONMENTS)} environment(s):\n')
    _results = []
    for _env in DATAVERSE_ENVIRONMENTS:
        ok, msg = _preflight_one(_env)
        _results.append(ok)
        print(f"  {'✓' if ok else '✗'} {_env}\n      {msg}")

    _passed = sum(_results)
    print(f'\n{_passed}/{len(DATAVERSE_ENVIRONMENTS)} passed.')
    if _passed == 0:
        raise SystemExit('✗ PREFLIGHT FAILED — no environment is reachable. Fix the messages above before ingesting.')
    elif _passed < len(DATAVERSE_ENVIRONMENTS):
        print('⚠ Some environments failed — cell 2a will SKIP those and ingest only the ones that passed.')
    else:
        print('✓ PREFLIGHT PASSED — safe to run cell 2a (full ingest).')


## 2. Ingest & load the source data  ·  *(branches by `SOURCE_MODE`)*

In [ ]:
# === 2a. INGEST — Dataverse Web API  *or*  Lakehouse re-parse  *or*  files ===
# Produces `tx` (pandas DataFrame of raw transcript rows) for cell 2b to parse.
# Branches by SOURCE_MODE so the rest of the notebook is source-agnostic.
import requests
import pandas as pd
from datetime import datetime, timezone, timedelta

def _dataverse_token(base):
    """Client-credentials token for one environment ({base}/.default)."""
    resp = requests.post(
        f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token',
        data={'grant_type': 'client_credentials', 'client_id': CLIENT_ID,
              'client_secret': CLIENT_SECRET, 'scope': f'{base.rstrip("/")}/.default'},
        timeout=60)
    resp.raise_for_status()
    return resp.json()['access_token']

def _fetch_transcripts(base):
    """Paged OData pull of conversationtranscripts from one environment; incremental when LOOKBACK_DAYS > 0."""
    base = base.rstrip('/')
    headers = {
        'Authorization':   f'Bearer {_dataverse_token(base)}',
        'Accept':          'application/json',
        'OData-MaxVersion':'4.0',
        'OData-Version':   '4.0',
        'Prefer':          f'odata.maxpagesize={PAGE_SIZE}',
    }
    select = ('conversationtranscriptid,content,conversationstarttime,schemaversion,'
              'schematype,name,metadata,statecode,statuscode,createdon')
    url = f'{base}/api/data/v9.2/conversationtranscripts?$select={select}'
    if LOOKBACK_DAYS and LOOKBACK_DAYS > 0:
        since = (datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%dT%H:%M:%SZ')
        url += f'&$filter=conversationstarttime ge {since}'
    rows, page = [], 0
    while url:
        r = requests.get(url, headers=headers, timeout=120)
        r.raise_for_status()
        body = r.json()
        rows.extend(body.get('value', []))
        url = body.get('@odata.nextLink')
        page += 1
        print(f'    page {page}: {len(rows):,} rows so far')
    return rows

tx = None

if SOURCE_MODE == 'dataverse':
    if not DATAVERSE_ENVIRONMENTS:
        raise SystemExit("✗ No valid environments. Set DATAVERSE_URL or DATAVERSE_URLS in CONFIG.")
    print(f'Ingesting conversationtranscripts from {len(DATAVERSE_ENVIRONMENTS)} environment(s)  '
          f'(LOOKBACK_DAYS={LOOKBACK_DAYS}, 0=full) ...\n')

    _frames = []
    for _env in DATAVERSE_ENVIRONMENTS:
        print(f'  {_env}')
        try:
            _raw = _fetch_transcripts(_env)
        except requests.HTTPError as e:
            code = e.response.status_code if e.response is not None else '?'
            print(f'    ⚠ SKIPPED ({code}) — {e}  (run the preflight to diagnose this environment)')
            continue
        _df = pd.DataFrame(_raw).fillna('') if _raw else pd.DataFrame()
        if len(_df):
            _df = _df.rename(columns={c: c.lower() for c in _df.columns})
            _df = _df[[c for c in _df.columns if not c.startswith('@') and not c.startswith('_')]]
            if TAG_ENVIRONMENT:
                _df['SourceEnvironment'] = _env
        print(f'    pulled: {len(_df):,}')
        if len(_df):
            _frames.append(_df)

    tx = pd.concat(_frames, ignore_index=True) if _frames else pd.DataFrame()
    print(f'\nTotal conversationtranscripts pulled across environments: {len(tx):,}')
    if RAW_TABLE and len(tx):
        (spark.createDataFrame(tx.astype(str))
              .write.mode(WRITE_MODE).option('overwriteSchema', 'true')
              .format('delta').saveAsTable(RAW_TABLE))
        print(f'  raw landed -> {RAW_TABLE} ({WRITE_MODE})')

elif SOURCE_MODE == 'lakehouse':
    print(f"Re-parsing from Lakehouse: {RAW_TABLE_INPUT}")
    if not spark.catalog.tableExists(RAW_TABLE_INPUT):
        raise SystemExit(f"✗ Table {RAW_TABLE_INPUT} does not exist. Run SOURCE_MODE='dataverse' first to land it.")
    _sdf = spark.table(RAW_TABLE_INPUT)
    _sdf = _sdf.toDF(*[c.lower() for c in _sdf.columns])
    tx = _sdf.toPandas().fillna('')
    print(f'  loaded {len(tx):,} raw transcript rows ({len(tx.columns)} cols)')

elif SOURCE_MODE == 'files':
    print(f"SOURCE_MODE='files' — transcripts will be read from {TRANSCRIPTS_FILE} in cell 2b.")

else:
    raise SystemExit(f"✗ Unknown SOURCE_MODE: {SOURCE_MODE!r}. "
                     f"Must be one of: 'dataverse', 'lakehouse', 'files'.")


In [ ]:
# === 2b. PARSE JSON CONTENT ===
import json, re
from datetime import datetime, timezone
import pandas as pd

def _read_csv(path):
    return (spark.read
            .option('header', True)
            .option('multiLine', True)
            .option('escape', '"')
            .option('encoding', 'UTF-8')
            .csv(path))

# `tx` is populated by cell 2a when SOURCE_MODE='dataverse'.
# In 'files' mode (or if 2a was skipped) read the CSV here instead.
if SOURCE_MODE == 'files' or 'tx' not in dir() or tx is None:
    transcripts_sdf = _read_csv(TRANSCRIPTS_FILE)
    transcripts_sdf = transcripts_sdf.toDF(*[c.lower() for c in transcripts_sdf.columns])
    tx = transcripts_sdf.toPandas().fillna('')

print(f'conversationtranscripts rows: {len(tx):,}')

content_col = next((c for c in tx.columns if c.lower().endswith('content')), None)
if content_col is None and len(tx) == 0:
    print('  conversationtranscripts returned 0 rows -> writing empty agent tables (schema preserved).')
elif content_col is None:
    raise RuntimeError('No content column found in conversationtranscripts')

def _safe_json(v):
    if not v:
        return None
    try:
        return json.loads(v)
    except Exception:
        return None

if content_col is None:
    parsed = tx.iloc[0:0].copy()
    parsed['content_json'] = pd.Series([], dtype=object)
else:
    tx['content_json'] = tx[content_col].map(_safe_json)
    parsed = tx[tx['content_json'].notna()].copy()
print(f'rows with valid JSON content: {len(parsed):,}')


## 3. Parsing helpers  ·  *(includes enrichment helpers used by cell 5c)*

In [ ]:
def keep_user_id(v):
    """Return the raw AAD objectId unchanged so users stay joinable downstream.

    NOTE: despite the legacy output column name ``user_id_hash``, this value is NOT
    hashed - it is the raw user id, kept intentionally for cross-source joins.
    Empty -> None so joins stay clean.
    """
    if v is None or v == '':
        return None
    return str(v)

# Backwards-compatible alias (existing cells call hash_id); same pass-through, no hashing.
hash_id = keep_user_id

def get_activities(content):
    if isinstance(content, dict):
        for k in ('activities', 'Activities', 'transcript', 'messages'):
            if k in content and isinstance(content[k], list):
                return content[k]
    if isinstance(content, list):
        return content
    return []

def extract_agent_name(act_list):
    """Returns (primary_agent_schema, [connected_agent_schemas...])."""
    primary = None
    connected = []
    for a in act_list:
        if not isinstance(a, dict):
            continue
        v = a.get('value') or {}
        if not isinstance(v, dict):
            continue
        if a.get('valueType') == 'ConnectedAgentInitializeTraceData':
            sn = v.get('connectedAgentBotSchemaName') or v.get('botSchemaName')
            if sn and sn not in connected:
                connected.append(sn)
            if primary is None:
                primary = v.get('parentBotSchemaName')
        # Single-agent transcripts have no ConnectedAgent trace; pick up a botSchemaName anywhere.
        elif primary is None:
            bsn = v.get('botSchemaName') or v.get('parentBotSchemaName')
            if bsn:
                primary = bsn
    return primary, connected

def _verdict_from_value(val):
    if val is None:
        return None
    if isinstance(val, bool):
        return 'Positive' if val else 'Negative'
    if isinstance(val, (int, float)):
        return 'Positive' if val > 0 else 'Negative'
    s = str(val).strip().lower()
    if s in ('true','1','like','thumbsup','thumbs up','up','positive','helpful','yes','good','smile'):
        return 'Positive'
    if s in ('false','0','dislike','thumbsdown','thumbs down','down','negative','unhelpful','no','bad','frown'):
        return 'Negative'
    return None

def extract_submitted_feedback(act_list):
    """Scan every canonical schema position where a completed feedback verdict
    writes back into a transcript activity stream. Returns (verdict, comment)."""
    verdict = None
    comment = None
    for a in act_list:
        if not isinstance(a, dict):
            continue
        for rx in (a.get('reactionsAdded') or []):
            if isinstance(rx, dict):
                v = _verdict_from_value(rx.get('type') or rx.get('reactionType'))
                if v:
                    verdict = v
        cd = a.get('channelData') if isinstance(a.get('channelData'), dict) else {}
        fl = cd.get('feedbackLoop') if isinstance(cd.get('feedbackLoop'), dict) else {}
        if fl:
            v = _verdict_from_value(fl.get('value') or fl.get('reaction') or fl.get('vote'))
            if v:
                verdict = v
            comment = comment or fl.get('feedbackText') or fl.get('text')
        cf = cd.get('feedback') if isinstance(cd.get('feedback'), dict) else {}
        if cf:
            v = _verdict_from_value(cf.get('helpful') if 'helpful' in cf else cf.get('reaction'))
            if v:
                verdict = v
            comment = comment or cf.get('feedbackText') or cf.get('text')
        val = a.get('value') if isinstance(a.get('value'), dict) else {}
        if val:
            action = str(val.get('actionName') or a.get('name') or '').lower()
            fb = val.get('feedback') if isinstance(val.get('feedback'), dict) else None
            if fb or 'feedback' in action or 'submitaction' in action:
                src = fb or val
                v = _verdict_from_value(src.get('helpful') if 'helpful' in src else (src.get('reaction') or src.get('vote') or src.get('rating')))
                if v:
                    verdict = v
                comment = comment or src.get('feedbackText') or src.get('text')
    return verdict, comment

def _iso(ms):
    return datetime.fromtimestamp(ms/1000, tz=timezone.utc).isoformat() if ms else None

print('helpers loaded')


# === ENRICHMENT (drop-in) — paste BEFORE the "Write Delta tables" cell ===
# Adds 8 new Copilot Studio deep-dive metrics + a new agent_variables fact table
# WITHOUT modifying existing build cells. The variables and new columns are
# derived from `parsed` (already in scope) and joined onto the existing
# `sessions` and `agent_performance` dataframes.
#
# Roll back: delete this cell and the next.

# === Enrichment helpers (added) ============================================
def extract_conversation_end(act_list):
    """Explicit session outcome. Verified against the real Copilot Studio schema:
    there is no ConversationEnd* trace; the platform's outcome lives on the
    `SessionInfo` activity as value.outcome / value.outcomeReason. (Falls back to
    a ConversationEnd* trace if a future export ever emits one.)
    Returns (outcome, outcome_reason) or (None, None)."""
    if not act_list:
        return None, None
    # Preferred: SessionInfo.outcome / outcomeReason (the real schema)
    for a in reversed(act_list):
        if not isinstance(a, dict):
            continue
        if a.get('valueType') != 'SessionInfo':
            continue
        v = a.get('value') or {}
        if not isinstance(v, dict):
            continue
        outcome = v.get('outcome')
        reason = v.get('outcomeReason')
        if outcome or reason:
            return (str(outcome) if outcome else None,
                    str(reason) if reason else None)
    # Fallback: legacy ConversationEnd* trace, if present
    for a in reversed(act_list):
        if not isinstance(a, dict):
            continue
        vt = a.get('valueType') or ''
        if 'ConversationEnd' not in vt:
            continue
        v = a.get('value') or {}
        if not isinstance(v, dict):
            continue
        end_type = (v.get('conversationendtype') or v.get('ConversationEndType')
                    or v.get('endType') or v.get('outcome'))
        end_reason = (v.get('endReason') or v.get('reason')
                      or v.get('outcomeReason'))
        if end_type or end_reason:
            return (str(end_type) if end_type else None,
                    str(end_reason) if end_reason else None)
    return None, None

def extract_topic_events(act_list):
    """Extract ordered topic events. Verified against the real Copilot Studio
    schema: there are no TopicStart/TopicEnd traces -- the closest signal is
    `IntentRecognition` (each recognised intent is a topic the bot entered).
    A topic is treated as 'completed' if no ErrorTraceData follows it before the
    next intent. Returns list of dicts: [{topic, started_ms, ended_ms, completed, dwell_ms}].
    (Falls back to TopicStart/TopicEnd traces if a future export emits them.)"""
    starts = []  # (topic, ts)
    ends = []    # (topic, ts, completion_type)
    intents = []  # (topic, ts) from IntentRecognition, in order
    for a in act_list or []:
        if not isinstance(a, dict):
            continue
        vt = a.get('valueType') or ''
        ts = a.get('timestampMs')
        v = a.get('value') or {}
        if not isinstance(v, dict):
            continue
        if vt == 'IntentRecognition':
            topic = v.get('intentTitle') or v.get('intentId')
            if topic:
                intents.append((topic, ts))
            continue
        topic = (v.get('topicName') or v.get('TopicName')
                 or v.get('topic') or v.get('intentTitle'))
        if 'TopicStart' in vt and topic and ts:
            starts.append((topic, ts))
        elif 'TopicEnd' in vt and topic and ts:
            ends.append((topic, ts, v.get('completionType') or v.get('endType')))
    # If no explicit Topic traces exist, derive topics from recognised intents.
    if not starts and intents:
        # detect whether any error trace occurred (used as a coarse completion flag)
        had_error = any(isinstance(a, dict) and a.get('valueType') == 'ErrorTraceData'
                        for a in (act_list or []))
        events = []
        for idx, (topic, ts) in enumerate(intents):
            nxt_ts = intents[idx + 1][1] if idx + 1 < len(intents) else None
            dwell = None
            try:
                if ts is not None and nxt_ts is not None:
                    dwell = int(nxt_ts) - int(ts)
            except (TypeError, ValueError):
                dwell = None
            events.append({
                'topic': topic,
                'started_ms': ts,
                'ended_ms': nxt_ts,
                'completed': (not had_error),
                'dwell_ms': dwell,
            })
        return events
    # pair them in order (first matching start, then end)
    paired = []
    used_ends = set()
    for topic, st in starts:
        match = None
        for j, (et_topic, et_ts, ctype) in enumerate(ends):
            if j in used_ends:
                continue
            if et_topic == topic and et_ts >= st:
                match = (j, et_ts, ctype)
                break
        if match:
            used_ends.add(match[0])
            paired.append({'topic': topic, 'started_ms': st,
                           'ended_ms': match[1], 'dwell_ms': match[1] - st,
                           'completed': True, 'completion_type': match[2]})
        else:
            paired.append({'topic': topic, 'started_ms': st,
                           'ended_ms': None, 'dwell_ms': None,
                           'completed': False, 'completion_type': None})
    return paired

def extract_auth_flag(act_list):
    """True if the session contains an AuthenticationTraceData success, OR if
    any message carries an aadObjectId (a sign the user resolved to Entra)."""
    saw_auth_success = False
    saw_aad_object = False
    for a in act_list or []:
        if not isinstance(a, dict):
            continue
        vt = a.get('valueType') or ''
        if 'Authentication' in vt:
            v = a.get('value') or {}
            if isinstance(v, dict):
                if v.get('isAuthenticated') or v.get('result') == 'Success' \
                        or v.get('status') == 'Authenticated':
                    saw_auth_success = True
        frm = a.get('from') or {}
        if isinstance(frm, dict) and frm.get('aadObjectId'):
            saw_aad_object = True
    return bool(saw_auth_success or saw_aad_object)

def extract_llm_and_plugin_metrics(act_list):
    """Pull LLM token usage + plugin invocations from generative orchestration
    traces. Returns dict of counters."""
    llm_calls = 0
    prompt_tokens = 0
    completion_tokens = 0
    total_tokens = 0
    plugin_calls = 0
    plugin_successes = 0
    generative_responses = 0
    for a in act_list or []:
        if not isinstance(a, dict):
            continue
        vt = a.get('valueType') or ''
        v = a.get('value') or {}
        if not isinstance(v, dict):
            continue
        if 'LLMCall' in vt or 'LlmCall' in vt or 'GenerativeAnswer' in vt:
            llm_calls += 1
            # tokens often nested under 'usage' or directly on value
            usage = v.get('usage') if isinstance(v.get('usage'), dict) else v
            pt = usage.get('promptTokenCount') or usage.get('promptTokens') or 0
            ct = usage.get('completionTokenCount') or usage.get('completionTokens') or 0
            tt = usage.get('totalTokenCount') or usage.get('totalTokens') or (pt + ct)
            try:
                prompt_tokens += int(pt or 0)
                completion_tokens += int(ct or 0)
                total_tokens += int(tt or 0)
            except (TypeError, ValueError):
                pass
            if 'GenerativeAnswer' in vt:
                generative_responses += 1
        # Real Copilot Studio schema: generative answers surface as a GPTAnswer
        # trace (no token usage is emitted). Count these as generative responses.
        if vt == 'GPTAnswer':
            generative_responses += 1
        if 'Plugin' in vt or 'Tool' in vt or 'ConnectorAction' in vt:
            plugin_calls += 1
            ok = v.get('success')
            if ok is None:
                ok = v.get('status') in ('Success', 'Succeeded', 'Ok')
            if ok:
                plugin_successes += 1
    return {
        'llm_call_count': llm_calls,
        'prompt_tokens': prompt_tokens,
        'completion_tokens': completion_tokens,
        'total_tokens': total_tokens,
        'plugin_call_count': plugin_calls,
        'plugin_success_count': plugin_successes,
        'generative_response_count': generative_responses,
    }

def extract_all_variables(act_list):
    """Collect variable name->value from VariableAssignment trace activities.
    Verified against the real Copilot Studio transcript schema: each assignment
    is its own activity (valueType == 'VariableAssignment') with value keys
    {name, id, newValue, type} -- there is no nested `variables` dict. Later
    activities win on conflict (most-recent value). Returns dict[str, Any] or {}."""
    out = {}
    for a in act_list or []:
        if not isinstance(a, dict):
            continue
        if a.get('valueType') != 'VariableAssignment':
            continue
        v = a.get('value') if isinstance(a.get('value'), dict) else None
        if not v:
            continue
        name = v.get('name')
        if name is None:
            continue
        out[str(name)] = v.get('newValue')
    return out

def categorise_error(error_code, is_user_error=False):
    """Bucket an error code into a coarse category for the Studio admin view."""
    if not error_code:
        return 'Unknown'
    code = str(error_code).lower()
    if is_user_error:
        return 'User'
    if any(k in code for k in ('knowledge', 'search', 'index', 'citation')):
        return 'Knowledge'
    if any(k in code for k in ('flow', 'workflow', 'connector', 'plugin', 'tool')):
        return 'Flow/Plugin'
    if any(k in code for k in ('auth', 'permission', 'forbidden', 'unauthor')):
        return 'Auth'
    if any(k in code for k in ('llm', 'generative', 'model', 'token', 'rate')):
        return 'LLM/Model'
    if any(k in code for k in ('timeout', 'transient', 'throttle')):
        return 'Transient'
    return 'Runtime'

print('enrichment helpers loaded')


## 4. Build per-session fact table — `agent_sessions`

In [ ]:
def build_sessions(parsed):
    rows = []
    for _, r in parsed.iterrows():
        acts = get_activities(r['content_json'])
        msgs = [a for a in acts if isinstance(a, dict) and a.get('type') == 'message']
        user_msgs = [a for a in msgs if (a.get('from') or {}).get('role') in (0, 'user')]
        bot_msgs  = [a for a in msgs if (a.get('from') or {}).get('role') in (1, 'bot')]
        user_id_raw = None
        for a in msgs:
            if (a.get('from') or {}).get('role') in (0, 'user'):
                user_id_raw = (a.get('from') or {}).get('aadObjectId') or (a.get('from') or {}).get('id')
                break
        primary_agent, connected_agents = extract_agent_name(acts)
        cost = latency = plan_steps = 0
        for a in acts:
            v = a.get('value') if isinstance(a, dict) else None
            if isinstance(v, dict) and 'displayedCost' in v:
                try:
                    cost += int(v.get('displayedCost') or 0)
                except (TypeError, ValueError):
                    pass
                plan_steps += 1
                et = v.get('executionTime')
                if et:
                    m = re.match(r'(\d+):(\d+):(\d+\.?\d*)', str(et))
                    if m:
                        h, mn, s = int(m.group(1)), int(m.group(2)), float(m.group(3))
                        latency += int((h*3600 + mn*60 + s) * 1000)
        k_searched = k_answered = False
        k_sources = 0
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'KnowledgeTraceData':
                v = a.get('value') or {}
                if isinstance(v, dict):
                    if v.get('isKnowledgeSearched'):
                        k_searched = True
                    if v.get('completionState') == 'Answered':
                        k_answered = True
                    k_sources += len(v.get('citedKnowledgeSources') or [])
        # --- grounding source: did the agent answer from internal knowledge, the web, or neither ---
        g_internal = g_external = False
        for a in acts:
            if not isinstance(a, dict):
                continue
            vt = a.get('valueType'); nm = str(a.get('name') or ''); gv = a.get('value') or {}
            if vt == 'KnowledgeTraceData' and isinstance(gv, dict) and gv.get('isKnowledgeSearched'):
                g_internal = True
            if 'UniversalSearch' in nm and isinstance(gv, dict):
                if gv.get('knowledgeSources'):
                    g_internal = True
                for ep in ((gv.get('searchContextTraceInfo') or {}).get('endpoints') or []):
                    h = str(ep).lower()
                    if 'bing.com' in h or 'bing.net' in h:
                        g_external = True
                    elif 'sharepoint' in h or 'onedrive' in h or 'crm.dynamics' in h or 'graph.microsoft' in h:
                        g_internal = True
                    elif h.startswith('http'):
                        g_external = True
            if 'web' in nm.lower() and 'search' in nm.lower():
                g_external = True
        grounding_source = ('Mixed (Internal+Web)' if g_internal and g_external
                            else 'External (Web)' if g_external
                            else 'Internal' if g_internal
                            else 'Ungrounded')
        errors = [ (a.get('value') or {}).get('errorCode') or 'Unknown'
                   for a in acts if isinstance(a, dict) and a.get('valueType') == 'ErrorTraceData' ]
        feedback_offered = sum(1 for a in acts if isinstance(a, dict)
                               and isinstance(a.get('channelData'), dict)
                               and 'feedbackLoop' in a.get('channelData', {}))
        verdict, comment = extract_submitted_feedback(acts)
        first_ts = last_ts = None
        for a in acts:
            ts = a.get('timestampMs') if isinstance(a, dict) else None
            if ts:
                first_ts = ts if first_ts is None or ts < first_ts else first_ts
                last_ts  = ts if last_ts  is None or ts > last_ts  else last_ts
        duration_ms = (last_ts - first_ts) if first_ts and last_ts else None
        session_start = _iso(first_ts) or r.get('conversationstarttime', '')
        first_prompt = ''
        for a in user_msgs:
            if a.get('text', ''):
                first_prompt = a['text'][:TEXT_TRUNCATE]
                break
        rows.append({
            'conversation_id': r.get('conversationtranscriptid', ''),
            'session_start_utc': session_start,
            'session_duration_ms': duration_ms,
            'user_id_hash': hash_id(user_id_raw),
            'primary_agent_schema': primary_agent,
            'connected_agent_schemas': '|'.join(connected_agents) if connected_agents else None,
            'connected_agent_count': len(connected_agents),
            'msg_count': len(msgs),
            'user_msg_count': len(user_msgs),
            'bot_msg_count': len(bot_msgs),
            'plan_step_count': plan_steps,
            'total_displayed_cost': cost,
            'total_latency_ms': latency,
            'knowledge_searched': k_searched,
            'knowledge_answered': k_answered,
            'knowledge_sources_count': k_sources,
            'grounding_source': grounding_source,
            'error_count': len(errors),
            'first_error_code': errors[0] if errors else None,
            'feedback_offered_count': feedback_offered,
            'feedback_submitted': 1 if verdict else 0,
            'feedback_verdict': verdict,
            'feedback_comment': comment,
            'first_user_prompt': first_prompt,
        })
    return pd.DataFrame(rows, columns=['conversation_id', 'session_start_utc', 'session_duration_ms', 'user_id_hash', 'primary_agent_schema', 'connected_agent_schemas', 'connected_agent_count', 'msg_count', 'user_msg_count', 'bot_msg_count', 'plan_step_count', 'total_displayed_cost', 'total_latency_ms', 'knowledge_searched', 'knowledge_answered', 'knowledge_sources_count', 'grounding_source', 'error_count', 'first_error_code', 'feedback_offered_count', 'feedback_submitted', 'feedback_verdict', 'feedback_comment', 'first_user_prompt'])

sessions = build_sessions(parsed)
print(f'sessions: {len(sessions):,}')


## 5. Build per-turn fact table — `agent_turns`

In [ ]:
def build_turns(parsed):
    rows = []
    for _, r in parsed.iterrows():
        conv_id = r.get('conversationtranscriptid', '')
        acts = get_activities(r['content_json'])
        intent_by_reply, knowledge_by_reply = {}, {}
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'IntentRecognition':
                v = a.get('value') or {}
                rid = a.get('replyToId')
                if rid and isinstance(v, dict):
                    intent_by_reply[rid] = {
                        'intent_title': v.get('intentTitle'),
                        'intent_id': v.get('intentId'),
                        'intent_score': (v.get('intentScore') or {}).get('score'),
                    }
            if isinstance(a, dict) and a.get('valueType') == 'KnowledgeTraceData':
                v = a.get('value') or {}
                rid = a.get('replyToId')
                if rid and isinstance(v, dict):
                    knowledge_by_reply[rid] = {
                        'knowledge_searched': bool(v.get('isKnowledgeSearched')),
                        'knowledge_answered': v.get('completionState') == 'Answered',
                        'knowledge_sources': '|'.join(v.get('citedKnowledgeSources') or []),
                    }
        for a in acts:
            if not isinstance(a, dict) or a.get('type') != 'message':
                continue
            role = (a.get('from') or {}).get('role')
            role_label = 'user' if role in (0, 'user') else ('bot' if role in (1, 'bot') else 'other')
            ts_ms = a.get('timestampMs')
            from_id = (a.get('from') or {}).get('aadObjectId') or (a.get('from') or {}).get('id')
            msg_id = a.get('id')
            intent = intent_by_reply.get(msg_id, {})
            knowledge = knowledge_by_reply.get(msg_id, {})
            cd = a.get('channelData') or {}
            fb_offered = isinstance(cd, dict) and 'feedbackLoop' in cd
            turn_verdict, turn_comment = extract_submitted_feedback([a])
            rows.append({
                'conversation_id': conv_id,
                'turn_id': msg_id,
                'turn_timestamp_utc': _iso(ts_ms),
                'turn_role': role_label,
                'turn_channel': a.get('channelId'),
                'from_user_hash': hash_id(from_id) if role_label == 'user' else None,
                'text_preview_500': (a.get('text') or '')[:TEXT_TRUNCATE],
                'text_length': len(a.get('text') or ''),
                'intent_title': intent.get('intent_title'),
                'intent_id': intent.get('intent_id'),
                'intent_score': intent.get('intent_score'),
                'knowledge_searched': knowledge.get('knowledge_searched'),
                'knowledge_answered': knowledge.get('knowledge_answered'),
                'knowledge_sources': knowledge.get('knowledge_sources'),
                'feedback_offered': fb_offered,
                'feedback_verdict': turn_verdict,
                'feedback_comment': turn_comment,
            })
    return pd.DataFrame(rows, columns=['conversation_id', 'turn_id', 'turn_timestamp_utc', 'turn_role', 'turn_channel', 'from_user_hash', 'text_preview_500', 'text_length', 'intent_title', 'intent_id', 'intent_score', 'knowledge_searched', 'knowledge_answered', 'knowledge_sources', 'feedback_offered', 'feedback_verdict', 'feedback_comment'])

turns = build_turns(parsed)
print(f'turns: {len(turns):,}')


## 5b. Enrich turns — topic tagging + role inference  ·  *(non-destructive)*

In [ ]:
# === 5b. ENRICH: topic tagging + role inference (non-destructive) ===
import re as _re
from collections import Counter as _Counter

# ---- clean Copilot Studio markup from preview text ----
_MARKUP = _re.compile(r'</?m-(?:official|faded)>|</?div>|</?br/?>|\*\*')
def _clean_text(t):
    return _MARKUP.sub(' ', t).strip() if isinstance(t, str) else ''

# ---- keyword topic taxonomy (operates on the FULL turn text) ----
# Ordered: noise buckets first, then business topics. First match wins.
_TOPIC_RULES = [
    ('\U0001F6AB Empty',            lambda m: m == ''),
    ('\U0001F6AB Command',          lambda m: m[:1] == '/'),
    ('\U0001F6AB Greeting',         lambda m: m in {'hi','hey','hello','test','ok','thanks','thank you','yes','no'}),
    ('\U0001F6AB Error / System',   lambda m: 'something unexpected happened' in m or 'error code' in m or 'systemerror' in m),
    ('\U0001F6AB Capability probe', lambda m: 'what can you do' in m or 'what this agent can do' in m or 'example of what' in m),
    ('Leave & Family',       ('leave','holiday','absence','paternity','maternity','carer','phased return','sabbatical')),
    ('Contracts & Policy',   ('redundan','notice period','medical retirement','contract','hybrid','classification','terms and condition')),
    ('Recruitment & Hiring', ('recruit','hiring','interview','candidate','vacancy',' cv','job role')),
    ('Onboarding & Joiner',  ('new joiner','onboard','induction','employment letter','basol','starter')),
    ('Access & IT Support',  ('password','iuser','login','log in','access','permission','mfa','vpn','email address','microphone','camera','laptop','device')),
    ('Pay & Bonus',          ('bonus','pay review','payslip','salary','expense','reward','uplift','amex')),
    ('Benefits & Perks',     ('benefit','pension','flex','cycle to work','discount','perk')),
    ('Field & Technical',    ('fibre','ethernet','ecc','cable','pole','manifold','construction','network')),
    ('Facilities & Workplace',('office','address','attendance','building','parking','desk')),
    ('Software & Licensing', ('licen','install','power bi','mural','software')),
    ('People & Directory',   ('who is','directory','contact number','org chart','manager')),
]
def classify_topic_keyword(text):
    m = _clean_text(text).lower()
    for label, rule in _TOPIC_RULES:
        if callable(rule):
            if rule(m):
                return label
        elif any(k in m for k in rule):
            return label
    return 'Other'

# ---- optional Azure OpenAI topic tagging (batched, fails safe to keyword) ----
def classify_topic_llm(texts):
    try:
        from openai import AzureOpenAI
    except Exception:
        print('  openai SDK unavailable -> keyword fallback'); return [classify_topic_keyword(t) for t in texts]
    if not AOAI_ENDPOINT or not AOAI_API_KEY:
        print('  AOAI not configured -> keyword fallback'); return [classify_topic_keyword(t) for t in texts]
    import json as _json
    client = AzureOpenAI(azure_endpoint=AOAI_ENDPOINT, api_key=AOAI_API_KEY, api_version=AOAI_API_VERSION)
    allowed = ', '.join(TOPIC_TAXONOMY); out = []; BATCH = 20
    for i in range(0, len(texts), BATCH):
        chunk = [(_clean_text(t)[:400] or '(empty)') for t in texts[i:i+BATCH]]
        prompt = ('Classify each numbered support message into EXACTLY ONE topic from: '
                  f'{allowed}. Reply JSON only: a list of {{"i":<index>,"topic":<topic>}}.\n'
                  + '\n'.join(f'{j}. {c}' for j, c in enumerate(chunk)))
        try:
            r = client.chat.completions.create(model=AOAI_DEPLOYMENT, temperature=0,
                messages=[{'role':'system','content':'You are a precise text classifier. Output JSON only.'},
                          {'role':'user','content':prompt}])
            txt = r.choices[0].message.content
            data = _json.loads(txt[txt.find('['): txt.rfind(']')+1])
            by_i = {d['i']: d['topic'] for d in data if d.get('topic') in TOPIC_TAXONOMY}
            out.extend(by_i.get(k, 'Other') for k in range(len(chunk)))
        except Exception as e:
            print(f'  LLM batch @{i} failed ({e}) -> keyword for this batch')
            out.extend(classify_topic_keyword(t) for t in texts[i:i+BATCH])
    return out

# ---- role inference from message content (diagnostic + opt-in correction) ----
def infer_role(row):
    raw = row.get('text_preview_500') or ''
    t = raw.strip(); low = _clean_text(t).lower()
    # system-injected error notices read as agent/system, regardless of channel attribution
    if low.startswith('sorry,') and ('error code' in low or 'unexpected happened' in low):
        return 'bot'
    agent_score = 0
    if '<m-official>' in raw or '<m-faded>' in raw: agent_score += 2
    if len(t) > 600: agent_score += 1
    if low.startswith(('how to','here','below','overview','step-by-step','summary of','official')): agent_score += 1
    if 'according to' in low or 'official documentation' in low: agent_score += 1
    user_score = 0
    if len(t) < 200: user_score += 1
    if t.endswith('?'): user_score += 1
    if low.startswith(('can you','do you','what','how do i','i need','please','where','is there','why','could you')): user_score += 1
    if agent_score > user_score: return 'bot'
    if user_score > agent_score: return 'user'
    return None  # undecided -> defer to raw turn_role

if 'turns' in dir() and not turns.empty:
    # 1) topic per turn
    if TOPIC_METHOD == 'llm':
        print('  Topic tagging: Azure OpenAI')
        turns['topic_derived'] = classify_topic_llm(list(turns['text_preview_500']))
    else:
        print('  Topic tagging: keyword taxonomy')
        turns['topic_derived'] = turns['text_preview_500'].map(classify_topic_keyword)

    # 2) role inference (non-destructive)
    if ROLE_INFERENCE:
        _inf = turns.apply(infer_role, axis=1)
        turns['role_inferred'] = _inf.where(_inf.notna(), turns['turn_role'])
        turns['role_mismatch'] = (turns['role_inferred'] != turns['turn_role'])
        _mm = int(turns['role_mismatch'].sum()); _tot = len(turns)
        print(f'  Role inference: {_mm:,}/{_tot:,} turns ({_mm/max(_tot,1):.0%}) differ from source role; original turn_role preserved.')
        # per-channel mismatch breakdown — shows WHERE the export mislabels roles
        try:
            _bych = turns.groupby('turn_channel')['role_mismatch'].mean().sort_values(ascending=False)
            print('  Mismatch rate by channel:')
            for ch, rate in _bych.items():
                print(f'    {str(ch):16} {rate:.0%}')
        except Exception:
            pass

    # 3) primary topic per session (most frequent non-noise topic across its turns)
    def _primary_topic(series):
        real = [x for x in series if isinstance(x, str) and not x.startswith('\U0001F6AB') and x != 'Other']
        return _Counter(real).most_common(1)[0][0] if real else 'Other'
    _by_conv = turns.groupby('conversation_id')['topic_derived'].apply(_primary_topic)
    if 'sessions' in dir() and not sessions.empty:
        sessions['primary_topic_derived'] = sessions['conversation_id'].map(_by_conv).fillna('Other')

    print(f"  Added topic_derived to {len(turns):,} turns"
          + (f" and primary_topic_derived to {len(sessions):,} sessions" if 'sessions' in dir() else "") + '.')
    print('\n  Top derived topics:')
    for label, n in turns['topic_derived'].value_counts().head(12).items():
        print(f'    {label:26} {n:>6,}')
else:
    print('  turns not built yet — run the build cells above first.')


## 6. Errors + sub-agent invocations — `agent_errors`, `agent_subagents`

In [ ]:
def build_errors(parsed):
    rows = []
    for _, r in parsed.iterrows():
        for a in get_activities(r['content_json']):
            if not isinstance(a, dict) or a.get('valueType') != 'ErrorTraceData':
                continue
            v = a.get('value') or {}
            rows.append({
                'conversation_id': r.get('conversationtranscriptid', ''),
                'error_timestamp_utc': _iso(a.get('timestampMs')),
                'error_code': v.get('errorCode'),
                'error_sub_code': v.get('errorSubCode'),
                'error_message': (v.get('errorMessage') or '')[:TEXT_TRUNCATE],
                'is_user_error': bool(v.get('isUserError')),
            })
    return pd.DataFrame(rows, columns=['conversation_id', 'error_timestamp_utc', 'error_code', 'error_sub_code', 'error_message', 'is_user_error'])

def build_subagents(parsed):
    rows = []
    for _, r in parsed.iterrows():
        for a in get_activities(r['content_json']):
            if not isinstance(a, dict):
                continue
            vt = a.get('valueType')
            if vt not in ('ConnectedAgentInitializeTraceData', 'ConnectedAgentCompletedTraceData'):
                continue
            v = a.get('value') or {}
            rows.append({
                'conversation_id': r.get('conversationtranscriptid', ''),
                'invocation_timestamp_utc': _iso(a.get('timestampMs')),
                'event_type': 'initialize' if vt.endswith('InitializeTraceData') else 'completed',
                'parent_agent_schema': v.get('parentBotSchemaName'),
                'connected_agent_schema': v.get('connectedAgentBotSchemaName') or v.get('botSchemaName'),
                'dialog_schema': v.get('dialogSchemaName'),
                'user_id_hash': hash_id(v.get('userId')),
                'plan_step_id': v.get('planStepId'),
            })
    return pd.DataFrame(rows, columns=['conversation_id', 'invocation_timestamp_utc', 'event_type', 'parent_agent_schema', 'connected_agent_schema', 'dialog_schema', 'user_id_hash', 'plan_step_id'])

errors = build_errors(parsed)
subagents = build_subagents(parsed)
print(f'errors: {len(errors):,}  |  sub-agent events: {len(subagents):,}')


## 7. Agent catalogue (dimension) — `agent_catalogue`

In [ ]:
AGENT_DESCRIPTIONS = {
    'copilots_header_3bd1a': 'M365 Copilot Orchestrator (header)',
    'msdyn_copilotforemployeeselfservicehr': 'Employee Self-Service HR',
    'msdyn_copilotforemployeeselfserviceit': 'Employee Self-Service IT',
    'msdyn_copilotforemployeeselfservicefinance': 'Employee Self-Service Finance',
    'msdyn_copilotforemployeeselfservicegeneral': 'Employee Self-Service General',
}

def build_agent_dim(sessions, subagents):
    s1 = pd.Series(sessions['primary_agent_schema'].dropna().unique())
    s2 = pd.Series(subagents['connected_agent_schema'].dropna().unique()) if not subagents.empty else pd.Series(dtype=str)
    s3 = pd.Series(subagents['parent_agent_schema'].dropna().unique()) if not subagents.empty else pd.Series(dtype=str)
    all_schemas = pd.concat([s1, s2, s3]).drop_duplicates()
    dim = pd.DataFrame({'agent_schema': sorted(all_schemas.tolist())})
    dim['agent_display_name'] = dim['agent_schema'].map(
        lambda s: AGENT_DESCRIPTIONS.get(s, s.replace('_', ' ').title()))
    def classify(s):
        s = s.lower()
        if 'header' in s or 'orchestrator' in s:
            return 'Orchestrator'
        if 'selfservice' in s or 'employeeself' in s:
            return 'Self-Service'
        if s.startswith('crb') or s.startswith('crd'):
            return 'Custom Declarative'
        if s.startswith('msdyn'):
            return 'First-Party Connected'
        return 'Other'
    dim['agent_class'] = dim['agent_schema'].map(classify)
    return dim

agent_dim = build_agent_dim(sessions, subagents)
print(f'unique agents: {len(agent_dim):,}')


## 8. Agent Performance KPI fact — `agent_performance`

In [ ]:
def build_agent_performance(parsed):
    rows = []
    for _, r in parsed.iterrows():
        acts = get_activities(r['content_json'])
        msgs = [a for a in acts if isinstance(a, dict) and a.get('type') == 'message']
        user_msgs = [a for a in msgs if (a.get('from') or {}).get('role') in (0, 'user')]
        bot_msgs  = [a for a in msgs if (a.get('from') or {}).get('role') in (1, 'bot')]
        primary_agent, connected_agents = extract_agent_name(acts)

        topics, last_intent_id = [], None
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'IntentRecognition':
                v = a.get('value') or {}
                t = v.get('intentTitle')
                if t and t not in topics:
                    topics.append(t)
                last_intent_id = v.get('intentId') or last_intent_id

        sys_error = user_error = False
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'ErrorTraceData':
                v = a.get('value') or {}
                if v.get('isUserError'):
                    user_error = True
                else:
                    sys_error = True

        latency_total = plan_steps = trace_count = event_count = 0
        for a in acts:
            if not isinstance(a, dict):
                continue
            if a.get('type') == 'trace':
                trace_count += 1
            elif a.get('type') == 'event':
                event_count += 1
            v = a.get('value') if isinstance(a.get('value'), dict) else None
            if v and 'executionTime' in v:
                plan_steps += 1
                m = re.match(r'(\d+):(\d+):(\d+\.?\d*)', str(v.get('executionTime')))
                if m:
                    h, mn, s = int(m.group(1)), int(m.group(2)), float(m.group(3))
                    latency_total += int((h*3600 + mn*60 + s) * 1000)
        avg_latency = int(latency_total / plan_steps) if plan_steps else 0

        first_ts = last_ts = None
        for a in acts:
            ts = a.get('timestampMs') if isinstance(a, dict) else None
            if ts:
                first_ts = ts if first_ts is None or ts < first_ts else first_ts
                last_ts  = ts if last_ts  is None or ts > last_ts  else last_ts
        duration_s = int((last_ts - first_ts) / 1000) if first_ts and last_ts else 0

        def _txt(a):
            return (a.get('text') or '')[:TEXT_TRUNCATE] if isinstance(a, dict) else ''
        first_user = _txt(user_msgs[0]) if user_msgs else ''
        last_user  = _txt(user_msgs[-1]) if user_msgs else ''
        first_bot  = _txt(bot_msgs[0]) if bot_msgs else ''
        last_bot   = _txt(bot_msgs[-1]) if bot_msgs else ''

        upn = None
        for a in acts:
            if not isinstance(a, dict):
                continue
            v = a.get('value') if isinstance(a.get('value'), dict) else {}
            vars_ = v.get('variables') if isinstance(v.get('variables'), dict) else {}
            upn = upn or vars_.get('ESS_UserContext_UPN') or vars_.get('Var_ESS_UserContext_UPN')

        # --- Heuristics ---
        engaged = bool(topics) or len(bot_msgs) > 0
        session_type = 'Engaged' if engaged else 'Unengaged'
        if connected_agents:
            outcome = 'Escalated'
        elif not engaged or (len(user_msgs) > 0 and len(bot_msgs) == 0) or (sys_error and len(bot_msgs) == 0):
            outcome = 'Abandoned'
        else:
            outcome = 'Resolved'
        outcome_reason = ('SystemError' if sys_error else 'UserError' if user_error else
                          ('Escalated' if outcome == 'Escalated' else
                           'Abandoned' if outcome == 'Abandoned' else 'Completed'))
        implied_success = 'TRUE' if (outcome == 'Resolved' and not sys_error) else 'FALSE'

        start_iso = _iso(first_ts) or r.get('conversationstarttime', '')
        end_iso = _iso(last_ts) or ''

        rows.append({
            'ConversationTranscriptId': r.get('conversationtranscriptid', ''),
            'ConversationStartTime': r.get('conversationstarttime', '') or start_iso,
            'SchemaVersion': r.get('schemaversion', ''),
            'BotName': primary_agent or '',
            'BotId': r.get('botid', ''),
            'AADTenantId': r.get('aadtenantid', ''),
            'BatchId': r.get('batchid', ''),
            'SessionStartTimeUtc': start_iso,
            'SessionEndTimeUtc': end_iso,
            'SessionType': session_type,
            'SessionOutcome': outcome,
            'TurnCount': len(msgs),
            'OutcomeReason': outcome_reason,
            'ImpliedSuccess': implied_success,
            'LastUserIntentId': last_intent_id,
            'IsDesignMode': 'FALSE',
            'Locale': r.get('locale', ''),
            'UserMessageCount': len(user_msgs),
            'BotMessageCount': len(bot_msgs),
            'TotalMessageCount': len(msgs),
            'FirstUserMessage': first_user,
            'LastUserMessage': last_user,
            'FirstBotMessage': first_bot,
            'LastBotMessage': last_bot,
            'TopicsTriggered': '|'.join(topics) if topics else '',
            'TopicCount': len(topics),
            'PrimaryTopic': topics[0] if topics else '',
            'DurationSeconds': duration_s,
            'AverageLatencyMs': avg_latency,
            'Var_ESS_UserContext_UPN': upn,
            'Var_ESS_Message_Disclaimer': '',
            'Var_ESS_UserContext_RefreshInterval_Hours': '',
            'Var_ESS_UserContext_Conversation_Initialized': '',
            'AllVariables': '',
            'TotalActivityCount': len(acts),
            'TraceActivityCount': trace_count,
            'EventActivityCount': event_count,
            'StatusCode': 1 if outcome == 'Resolved' else 0,
            'StateCode': 1,
        })
    return pd.DataFrame(rows, columns=['ConversationTranscriptId', 'ConversationStartTime', 'SchemaVersion', 'BotName', 'BotId', 'AADTenantId', 'BatchId', 'SessionStartTimeUtc', 'SessionEndTimeUtc', 'SessionType', 'SessionOutcome', 'TurnCount', 'OutcomeReason', 'ImpliedSuccess', 'LastUserIntentId', 'IsDesignMode', 'Locale', 'UserMessageCount', 'BotMessageCount', 'TotalMessageCount', 'FirstUserMessage', 'LastUserMessage', 'FirstBotMessage', 'LastBotMessage', 'TopicsTriggered', 'TopicCount', 'PrimaryTopic', 'DurationSeconds', 'AverageLatencyMs', 'Var_ESS_UserContext_UPN', 'Var_ESS_Message_Disclaimer', 'Var_ESS_UserContext_RefreshInterval_Hours', 'Var_ESS_UserContext_Conversation_Initialized', 'AllVariables', 'TotalActivityCount', 'TraceActivityCount', 'EventActivityCount', 'StatusCode', 'StateCode'])

agent_performance = build_agent_performance(parsed)
print(f'agent performance rows: {len(agent_performance):,}')


## 8c. Deep-dive enrichment — explicit outcome, topic dwell, LLM/plugin metrics, `agent_variables`  ·  *(non-destructive)*

In [ ]:
# === 5c. Deep-dive enrichment — explicit outcomes, topic dwell, LLM/plugin metrics, agent_variables ===
# NON-DESTRUCTIVE: derives new columns from `parsed` and JOINS them onto the
# existing `sessions` and `agent_performance` dataframes (no edits to build_*).
# Also builds the new `agent_variables` fact table and sets `is_returning_user`
# on sessions.

# ---- 1) Re-derive enrichment columns from `parsed` ----
def _enrich_per_session(parsed):
    """For every parsed transcript row, compute the new column values."""
    rows = []
    for _, r in parsed.iterrows():
        acts = get_activities(r['content_json'])
        end_type, end_reason = extract_conversation_end(acts)
        topic_events = extract_topic_events(acts)
        topics_started = len(topic_events)
        topics_completed = sum(1 for t in topic_events if t['completed'])
        topic_completion_rate = (topics_completed / topics_started) if topics_started else 0.0
        primary_topic_dwell = (topic_events[0]['dwell_ms'] or 0) if topic_events else 0
        total_topic_dwell = sum((t['dwell_ms'] or 0) for t in topic_events)
        is_authenticated = extract_auth_flag(acts)
        llm = extract_llm_and_plugin_metrics(acts)
        # multi-agent: any ConnectedAgent initialize trace
        multi_agent = any(isinstance(a, dict) and a.get('valueType') == 'ConnectedAgentInitializeTraceData'
                          for a in acts)
        # first error category
        first_err_code, first_err_user = None, False
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'ErrorTraceData':
                v = a.get('value') or {}
                first_err_code = v.get('errorCode')
                first_err_user = bool(v.get('isUserError'))
                break
        rows.append({
            'conversation_id': r.get('conversationtranscriptid', ''),
            'session_outcome_explicit': end_type,
            'session_outcome_reason_explicit': end_reason,
            'is_authenticated': is_authenticated,
            'topics_started': topics_started,
            'topics_completed': topics_completed,
            'topic_completion_rate': topic_completion_rate,
            'primary_topic_dwell_ms': primary_topic_dwell,
            'total_topic_dwell_ms': total_topic_dwell,
            'llm_call_count': llm['llm_call_count'],
            'prompt_token_count': llm['prompt_tokens'],
            'completion_token_count': llm['completion_tokens'],
            'total_token_count': llm['total_tokens'],
            'plugin_call_count': llm['plugin_call_count'],
            'plugin_success_count': llm['plugin_success_count'],
            'generative_response_count': llm['generative_response_count'],
            'multi_agent_session': multi_agent,
            'first_error_code_enr': first_err_code,
            'error_category': categorise_error(first_err_code, first_err_user) if first_err_code else None,
        })
    return pd.DataFrame(rows)

_enr = _enrich_per_session(parsed)
print(f'enrichment columns derived: {len(_enr):,} rows, {len(_enr.columns)-1} new columns')

# ---- 2) Merge enrichment columns into existing sessions ----
_session_enr = _enr[['conversation_id', 'session_outcome_explicit',
                     'session_outcome_reason_explicit', 'is_authenticated']].copy()
sessions = sessions.merge(_session_enr, on='conversation_id', how='left')
print(f'sessions: now {len(sessions.columns)} columns (added 3 enrichment cols)')

# ---- 3) Merge ALL enrichment columns into agent_performance ----
# agent_performance uses 'ConversationTranscriptId' as its key
_perf_enr = _enr.copy()
_perf_enr = _perf_enr.rename(columns={
    'conversation_id': 'ConversationTranscriptId',
    'session_outcome_explicit': 'SessionOutcomeExplicit',
    'session_outcome_reason_explicit': 'SessionOutcomeReasonExplicit',
    'is_authenticated': 'IsAuthenticated',
    'topics_started': 'TopicsStarted',
    'topics_completed': 'TopicsCompleted',
    'topic_completion_rate': 'TopicCompletionRate',
    'primary_topic_dwell_ms': 'PrimaryTopicDwellMs',
    'total_topic_dwell_ms': 'TotalTopicDwellMs',
    'llm_call_count': 'LLMCallCount',
    'prompt_token_count': 'PromptTokenCount',
    'completion_token_count': 'CompletionTokenCount',
    'total_token_count': 'TotalTokenCount',
    'plugin_call_count': 'PluginCallCount',
    'plugin_success_count': 'PluginSuccessCount',
    'generative_response_count': 'GenerativeResponseCount',
    'multi_agent_session': 'MultiAgentSession',
    'first_error_code_enr': 'FirstErrorCode',
    'error_category': 'ErrorCategory',
})
agent_performance = agent_performance.merge(_perf_enr, on='ConversationTranscriptId', how='left')
print(f'agent_performance: now {len(agent_performance.columns)} columns (added 18 enrichment cols)')

# ---- 4) Build agent_variables fact table ----
def _build_variables(parsed):
    rows = []
    for _, r in parsed.iterrows():
        acts = get_activities(r['content_json'])
        cid = r.get('conversationtranscriptid', '')
        vars_ = extract_all_variables(acts)
        for name, value in vars_.items():
            if value is None:
                vstr = None
            elif isinstance(value, (dict, list)):
                try:    vstr = json.dumps(value)[:1000]
                except: vstr = str(value)[:1000]
            else:
                vstr = str(value)[:1000]
            rows.append({'conversation_id': cid, 'variable_name': name, 'variable_value': vstr})
    return pd.DataFrame(rows, columns=['conversation_id', 'variable_name', 'variable_value'])

variables = _build_variables(parsed)
print(f'variables: {len(variables):,} rows ({variables["variable_name"].nunique() if len(variables) else 0:,} distinct names)')

# ---- 5) Returning-user flag (post-build) on sessions ----
if 'user_id_hash' in sessions.columns and not sessions.empty:
    _s = sessions.sort_values('session_start_utc').copy()
    _seen, flags = {}, []
    for uid in _s['user_id_hash']:
        if uid is None or pd.isna(uid):
            flags.append(False); continue
        flags.append(uid in _seen)
        _seen[uid] = True
    _s['is_returning_user'] = flags
    sessions = _s.sort_index()
    print(f'is_returning_user set: {int(sessions["is_returning_user"].sum()):,} of {len(sessions):,}')


## 9. Write Delta tables  ·  *(7 tables, with 0-row safeguard)*

In [ ]:
# === 9. Write Delta tables ===

# --- ZERO-ROW SAFEGUARD -----------------------------------------------------
# If the ingest+parse produced 0 rows of `parsed`, the build_* functions returned
# empty dataframes. Writing those over the live Lakehouse would wipe a good
# snapshot. Bail out BEFORE any DROP/overwrite, with a clear message.
_n_parsed = len(parsed) if 'parsed' in dir() else 0
if ABORT_ON_EMPTY and _n_parsed == 0:
    raise SystemExit(
        f"✗ ABORTING WRITE — `parsed` is empty (0 rows). The build_* outputs would overwrite "
        f"existing Lakehouse tables with zero rows. Common causes:\n"
        f"  - Dataverse Application User has a USER-scope read role on conversationtranscript "
        f"    (should be ORGANIZATION-scope)\n"
        f"  - LOOKBACK_DAYS window has no transcripts in any environment\n"
        f"  - SOURCE_MODE='lakehouse' and {RAW_TABLE_INPUT} is empty\n"
        f"Set ABORT_ON_EMPTY=False in CONFIG only if you want to intentionally reset all tables to empty."
    )

# --- Carry SourceEnvironment through to the transcript-derived tables -------
_has_env = 'SourceEnvironment' in parsed.columns
if _has_env:
    _conv_env = dict(zip(parsed.get('conversationtranscriptid', pd.Series(dtype=str)),
                         parsed['SourceEnvironment']))
    def _add_env(df, key):
        if df is not None and not df.empty and key in df.columns:
            df = df.copy()
            df['source_environment'] = df[key].map(lambda c: _conv_env.get(c))
        return df
    sessions          = _add_env(sessions,          'conversation_id')
    turns             = _add_env(turns,             'conversation_id')
    errors            = _add_env(errors,            'conversation_id')
    subagents         = _add_env(subagents,         'conversation_id')
    agent_performance = _add_env(agent_performance, 'ConversationTranscriptId')
    variables         = _add_env(variables,         'conversation_id')
    if not agent_dim.empty and not sessions.empty and 'source_environment' in sessions.columns:
        _env_by_agent = (sessions.dropna(subset=['primary_agent_schema'])
                         .groupby('primary_agent_schema')['source_environment']
                         .apply(lambda s: '|'.join(sorted({x for x in s if x}))))
        agent_dim = agent_dim.copy()
        agent_dim['source_environments'] = agent_dim['agent_schema'].map(_env_by_agent).fillna('')
    print(f'  SourceEnvironment carried through ({len(set(v for v in _conv_env.values() if v))} environment(s)).')
else:
    print('  No SourceEnvironment column (files/lakehouse mode or untagged) — tables written without it.')

outputs = {
    'agent_sessions':     sessions,
    'agent_turns':        turns,
    'agent_errors':       errors,
    'agent_subagents':    subagents,
    'agent_catalogue':    agent_dim,
    'agent_performance':  agent_performance,
    'agent_variables':    variables,
}

_ws = re.compile(r'[\r\n\t]+')
def _flatten(v):
    return _ws.sub(' ', v).strip() if isinstance(v, str) else v

from pyspark.sql.types import StructType, StructField, StringType

MERGE_KEYS = {
    'agent_sessions':     ['conversation_id'],
    'agent_turns':        ['turn_id'],
    'agent_errors':       ['conversation_id', 'error_timestamp_utc', 'error_code'],
    'agent_subagents':    ['conversation_id', 'invocation_timestamp_utc', 'connected_agent_schema', 'plan_step_id'],
    'agent_catalogue':    ['agent_schema'],
    'agent_performance':  ['ConversationTranscriptId'],
    'agent_variables':    ['conversation_id', 'variable_name'],
}

def _merge_delta(sdf, name, table):
    keys = list(MERGE_KEYS.get(name, []))
    if 'source_environment' in sdf.columns and name != 'agent_catalogue' and 'source_environment' not in keys:
        keys.append('source_environment')
    from delta.tables import DeltaTable
    if not keys or not spark.catalog.tableExists(table):
        (sdf.write.mode('append')
            .option('mergeSchema', 'true')
            .option('delta.columnMapping.mode', 'name')
            .option('delta.minReaderVersion', '2')
            .option('delta.minWriterVersion', '5')
            .format('delta').saveAsTable(table))
        return 'append(first-load)'
    cond = ' AND '.join([f't.`{k}` <=> s.`{k}`' for k in keys])
    (DeltaTable.forName(spark, table).alias('t')
        .merge(sdf.alias('s'), cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    return f'merge(on {"+".join(keys)})'

for name, pdf in outputs.items():
    pdf = pdf.copy()
    for col in pdf.columns:
        pdf[col] = pdf[col].map(_flatten)
    pdf = pdf.astype(object).where(pd.notna(pdf), None)
    schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
    sdf = spark.createDataFrame(pdf.astype(str).where(pd.notna(pdf), None), schema=schema)
    table = f'{OUTPUT_PREFIX}.{name}'
    if WRITE_MODE == 'merge':
        _how = _merge_delta(sdf, name, table)
        print(f'  {table:28} {len(pdf):>7,} rows  ->  {_how}')
    else:
        if WRITE_MODE == 'overwrite':
            spark.sql(f'DROP TABLE IF EXISTS {table}')
        (sdf.write.mode(WRITE_MODE)
            .option('overwriteSchema', 'true')
            .option('delta.columnMapping.mode', 'name')
            .option('delta.minReaderVersion', '2')
            .option('delta.minWriterVersion', '5')
            .format('delta').saveAsTable(table))
        print(f'  {table:28} {len(pdf):>7,} rows  ->  written ({WRITE_MODE})')

print('\nDone. Seven Delta tables written. Refresh the PBIP / Direct Lake semantic model to pick them up.')
